<a href="https://colab.research.google.com/github/spg3098-alt/redrob-ranker/blob/master/demo_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Redrob Ranker — Demo Sandbox

This notebook demonstrates the **Redrob Intelligent Candidate Discovery** ranker on a 50-candidate sample.

It clones the repo, installs dependencies, and runs the full ranking pipeline end-to-end — CPU only, no network calls during ranking, no GPU required.

**Runtime:** ~15 seconds for 50 candidates. Full 100K run takes ~90s on CPU.

In [ ]:
# Step 1: Clone the repo
!git clone https://github.com/spg3098-alt/redrob-ranker.git
%cd redrob-ranker

Cloning into 'redrob-ranker'...
remote: Enumerating objects: 72, done.
remote: Counting objects: 100% (72/72), done.
remote: Compressing objects: 100% (37/37), done.
remote: Total 72 (delta 31), reused 68 (delta 27), pack-reused 0 (from 0)
Receiving objects: 100% (72/72), 101.27 KiB | 1.37 MiB/s, done.
Resolving deltas: 100% (31/31), done.
/content/redrob-ranker


In [ ]:
# Step 2: Install dependencies
!pip install -q -r requirements.txt

In [ ]:
# Step 3: Run the ranker on the 50-candidate demo sample
# sample_candidates.jsonl is pre-bundled in the repo
!python rank.py --candidates ./sample_candidates.jsonl --out ./demo_output.csv

Traceback (most recent call last):
  File "/content/redrob-ranker/rank.py", line 69, in <module>
    main()
  File "/content/redrob-ranker/rank.py", line 34, in main
    for c in load.stream_candidates(args.candidates):
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/redrob-ranker/src/redrob_ranker/load.py", line 19, in stream_candidates
    yield json.loads(line)
          ^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/json/__init__.py", line 335, in loads
    raise JSONDecodeError("Unexpected UTF-8 BOM (decode using utf-8-sig)",
json.decoder.JSONDecodeError: Unexpected UTF-8 BOM (decode using utf-8-sig): line 1 column 1 (char 0)


## How it works

```
additive   = 0.22*title + 0.38*skills_trust + 0.14*semantic
           + 0.10*ml_yoe_experience + 0.08*location + 0.08*eval_text_evidence

gated      = additive
           × domain_mismatch_penalty
           × consulting_only_penalty
           × notice_multiplier          # x0.65-1.00 by notice days
           × ml_yoe_gate                # x0.25/<3y, x0.60/<4y, x0.95/<5y
           × core_coverage_gate         # x0.70 if <=1/4 must-haves covered
           × self_assessment_penalty    # x0.85 if summary admits weakness

final      = gated
           x behavioral_multiplier      # x0.40-1.0 (capped for low coverage)
           x honeypot_sink              # x0.03 for impossible profiles
```

See `src/redrob_ranker/` for full source. No LLM calls, no GPU, no network during ranking.